# Pre-lab 1: Failure Modes, Calibration, and the Equation Design
### ME 323 Module 1

<img src="https://raw.githubusercontent.com/andrewvoss8-boop/core-me-data-science-activities-public/main/me323/Module1_drafts/figures/I_beam_dimensions.jpg" alt="I-beam dimensions" width="260">

In the image, `b` is web thickness, `H` is web height, `B` is flange width,
and the image's `h` is the flange thickness called `t_f` in this notebook.

You will load the class beam data, compute strength-to-weight, code the
failure-mode formulas, inspect where they disagree with real failures, tune
the parameters by hand, calibrate them, and optimize the class equation design.

Lines marked `FILL IN` are yours. Each step prints a checkpoint.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

# Fixed geometry (you choose b and H_web; everything else is set)
B, TH, L = 10.0, 18.0, 150.0     # flange width, total height, test span (mm)
LP = 172.0                        # printed length (mm); overhangs the 150 mm span
KMASS = 0.2045                    # g/mm^2: mass per unit cross-section area at 172 mm

# Handbook starting values — Pre-lab 1 calibrates the three marked ones
SY = 76e6                         # Pa, PLA strength                 (calibrated)
K_LTB = 0.33                      # fixture effective-length factor  (calibrated)
CS = 1.0                          # web shear-strength multiplier    (calibrated)
E, G = 2.5e9, 2.5e9 / 2.6         # Young's / shear modulus (Pa) — fixed
C1, C2 = 1.35, 0.55               # LTB moment-gradient / load-height factors
print("Constants loaded. Nominal SY =", SY/1e6, "MPa")

## 1. The data

These are fourteen real three-point-bend tests. The beams were printed at
172 mm and tested on a 150 mm support span. `failure_note` is the technician's
observation, not a model-generated label. `weight_g` is measured; `mass_est_g`
comes from nominal geometry. They should not be identical. Read the notes and
the mass discrepancy before fitting anything.

In [ ]:
URL = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
       "core-me-data-science-activities-public/main/data/student_beams_B10_L150.csv")
try:
    df = pd.read_csv(URL); print("loaded from GitHub")
except Exception:
    df = pd.read_csv("student_beams_B10_L150.csv"); print("loaded local copy")
df = df.rename(columns={"b_mm": "b", "H_web_mm": "H"})

def estimated_mass_g(b, H):
    A = b * H + B * (TH - H)      # cross-section area, mm^2
    return KMASS * A              # grams
df["mass_est_g"] = estimated_mass_g(df.b, df.H)
df["mass_delta_g"] = df.weight_g - df.mass_est_g
df["mass_delta_pct"] = 100*df.mass_delta_g/df.mass_est_g
print(len(df), "tested beams")

pd.set_option("display.max_colwidth", None)
df[["beam_id", "b", "H", "weight_g", "mass_est_g", "mass_delta_pct",
    "strength_N", "failure_note"]].round({"mass_delta_pct": 1})

## 2. Strength-to-weight

The design objective uses estimated mass from nominal geometry:

$$A=bH+B(T-H), \qquad m_{est}=K_{mass}A.$$

Measured mass retains print variation, dimensional error, and scale error.
Estimated mass keeps the optimization objective available before printing.
Keep both. Frozen class rankings use `strength_N / mass_est_g`.

In [ ]:
df["str_to_weight"] = ____    # >>> FILL IN: strength divided by estimated mass
df["str_to_weight_measured_mass"] = df.strength_N / df.weight_g
top = df.sort_values("str_to_weight", ascending=False)
print(top[["beam_id", "b", "H", "weight_g", "mass_est_g",
           "str_to_weight_measured_mass", "str_to_weight"]]
      .round(2).to_string(index=False))
print("\nMeasured minus estimated mass:")
print(df[["beam_id", "mass_delta_g", "mass_delta_pct"]]
      .round(2).to_string(index=False))
print("\nCHECKPOINT: the best of the original 14 beams should be beam 4 "
      "(b=3.25, H_web=15.1) at 36.88 N/g.")
print("If you are not getting that, check your work or talk to a TA.")

## 3. Section geometry and what each property controls

The flange width `B = 10 mm` and total height `T = 18 mm` are fixed. You choose
web thickness `b` and web height `H`; therefore flange thickness is
$t_f=(T-H)/2$ and outer-fiber distance is the fixed $c=T/2$.

- $A=bH+B(T-H)$ is material area. It sets mass and the average-web-shear area.
  It is linear in either design variable separately, but bilinear jointly.
- $I_x=[BT^3-(B-b)H^3]/12$ is the strong-axis second moment. Bending stress and
  bending deflection scale as $1/I_x$. It grows linearly with `b`; because
  total height is fixed, it decreases cubically as `H` grows and the flanges thin.
- $I_y=[Hb^3+(T-H)B^3]/12$ is the weak-axis second moment. It controls lateral
  bending in LTB. Its web term is cubic in `b`, while increasing `H` replaces
  wide flange material with narrow web material and usually lowers $I_y$.
- $J$ is the St. Venant torsion constant. LTB resistance grows with it. It is
  dominated by thickness-cubed terms with aspect-ratio corrections.
- $C_w=I_y(H+t_f)^2/4$ is the warping constant used by the LTB approximation.
  Its dependence is more complex because $I_y$ falls while flange separation grows.

The code returns SI units. Inputs arrive in millimeters, but $I_x,I_y,J$ are in
$m^4$, $C_w$ is in $m^6$, and lengths in the returned dictionary are meters.

Modes not in the capacity model still depend on this geometry. Thin-plate shear
buckling scales roughly as $Et^3/H$ in load; flange local-buckling stress scales
roughly as $E(t_f/w)^2$, where $w=(B-b)/2$ is flange outstand. Web crippling also
depends on nose/support contact geometry that this dataset does not record.

In [ ]:
def section_props(b, H):
    """I-section properties. b, H in mm; everything returned in METERS/SI."""
    tf = (TH - H) / 2.0
    b_, h_, B_, tf_ = b/1e3, H/1e3, B/1e3, tf/1e3
    c = (TH/1e3) / 2
    Ix = (b_*h_**3)/12 + 2*((B_*tf_**3)/12 + B_*tf_*(h_/2 + tf_/2)**2)
    Iy = (h_*b_**3)/12 + 2*(tf_*B_**3)/12
    def J_rect(x, y):
        short, long = min(x, y), max(x, y)
        r = short/long
        beta = 1 - 0.63*r + 0.052*r**5
        return (1/3)*beta*long*short**3
    J = J_rect(b_, h_) + 2*J_rect(tf_, B_)
    Cw = Iy*(h_ + tf_)**2/4
    return dict(Ix=Ix, Iy=Iy, J=J, Cw=Cw, c=c,
                b=b_, h=h_, tf=tf_, B=B_)
section_props(2.0, 12.0)

## 4. Failure mode: flexural yield

<img src="https://raw.githubusercontent.com/andrewvoss8-boop/core-me-data-science-activities-public/main/me323/Module1_drafts/figures/flexural%20stress%20photo.webp" alt="Flexural stress distribution" width="520">

For a centered load in three-point bending,
$M_{max}=PL/4$. The outer flange fiber reaches yield when
$\sigma=M_{max}c/I_x=\sigma_y$:

$$P_{bend}=\frac{4\sigma_y I_x}{cL}.$$

This assumes Euler-Bernoulli bending, linear elasticity to first yield,
homogeneous isotropic material, and no local buckling or contact damage.

In [ ]:
def P_bend(p, sy):
    return ____    # >>> FILL IN: P_bend; section_props is SI but L is in mm

Pb_check = P_bend(section_props(2.0, 12.0), SY)
assert 750 < Pb_check < 920, (
    f"P_bend(2,12) = {Pb_check:.6g} N is off. Convert L from mm to m.")
print(f"P_bend(2,12) = {Pb_check:.1f} N   (checkpoint: about 835 N)")

## 5. Average web shear and the interaction surrogate

In either span between a support and the center load, internal shear force is
$V=P/2$. The current model divides that force by the entire web area:

$$\tau_{avg}=\frac{V}{bH}, \qquad
P_{shear}=2bH\frac{c_s\sigma_y}{\sqrt{3}}.$$

This is a whole-web average. It is not the shear stress at the flange root,
neutral axis, or any single material point.

The capacity model combines outer-fiber bending and average web shear through

$$\frac{1}{P_{int}^2}=\frac{1}{P_{bend}^2}+\frac{1}{P_{shear}^2}.$$

That algebra would follow from von Mises only if both stresses were evaluated
at the same point. They are not, and fitted `c_s` is not a material von Mises
constant. We therefore call this an empirical interaction surrogate.

In [ ]:
def P_shear(p, sy, cs):
    return ____    # >>> FILL IN: use p["b"] and p["h"] in meters
def P_interaction_surrogate(Pb, Ps):
    return ____    # >>> FILL IN: inverse-square interaction

def P_pointwise_yield(p, sy, n=801, return_detail=False):
    """Elastic first yield using co-located My/I and VQ/(It)."""
    c, h2 = p["c"], p["h"]/2
    eps = max(c, 1.0)*1e-10
    y = np.unique(np.r_[np.linspace(0, c, n),
                        max(0, h2-eps), min(c, h2+eps)])
    in_web = y <= h2
    width = np.where(in_web, p["b"], p["B"])
    q_flange = p["B"]*p["tf"]*(h2 + p["tf"]/2)
    Q = np.where(in_web,
                 q_flange + p["b"]*(h2-y)*(y+h2)/2,
                 p["B"]*(c-y)*(y+c)/2)
    sigma_per_N = (L/1e3)*y/(4*p["Ix"])
    tau_per_N = Q/(2*p["Ix"]*width)
    vm_per_N = np.sqrt(sigma_per_N**2 + 3*tau_per_N**2)
    loads = np.divide(sy, vm_per_N, out=np.full_like(vm_per_N, np.inf),
                      where=vm_per_N > 0)
    i = int(np.argmin(loads))
    if return_detail:
        return float(loads[i]), float(y[i]), float(sigma_per_N[i]), float(tau_per_N[i])
    return float(loads[i])

p = section_props(2.0, 12.0)
Ps_check = P_shear(p, SY, CS)
Pint_check = P_interaction_surrogate(P_bend(p, SY), Ps_check)
Ppoint_check, ycrit, _, _ = P_pointwise_yield(p, SY, return_detail=True)
assert 1900 < Ps_check < 2320, f"P_shear(2,12) = {Ps_check:.6g} N is off."
assert Pint_check < min(P_bend(p, SY), Ps_check)
print(f"average-web shear limit = {Ps_check:.1f} N")
print(f"interaction surrogate   = {Pint_check:.1f} N   (checkpoint: about 776 N)")
print(f"pointwise first yield    = {Ppoint_check:.1f} N at y={ycrit*1e3:.2f} mm")
print("The surrogate stays in capacity() for continuity; the pointwise value is the theoretical check.")

## 6. Lateral-torsional buckling

LTB is provided because it carries the most assumptions: elastic warping,
top-flange loading, idealized supports, initial straightness, and an effective
unbraced length $L_b=kL$. Here `k` represents fixture restraint, not a beam
material property. The expression is capped at the flexural-yield moment.

In [ ]:
def P_LTB(p, sy, k):
    My = sy*p["Ix"]/p["c"]
    Lb, zg = k*L/1e3, p["c"]
    R = p["Cw"]/p["Iy"] + (Lb**2*G*p["J"])/(np.pi**2*E*p["Iy"]) + (C2*zg)**2
    Mcr = C1*np.pi**2*E*p["Iy"]/Lb**2 * (np.sqrt(R) - C2*zg)
    return 4*min(My, Mcr)/(L/1e3)

def capacity(b, H, sy, k, cs):
    """Empirical interaction surrogate capped by the LTB prediction."""
    p = section_props(b, H)
    Pint = P_interaction_surrogate(P_bend(p, sy), P_shear(p, sy, cs))
    return min(Pint, P_LTB(p, sy, k))

def gov_mode(b, H, sy, k, cs):
    """Dominant pure-mode proxy, not an observed failure-mechanism label."""
    p = section_props(b, H)
    Pb, Ps, Pl = P_bend(p, sy), P_shear(p, sy, cs), P_LTB(p, sy, k)
    if Ps < min(Pb, Pl):
        return "interaction/shear proxy"
    return "LTB" if Pl < 0.999*Pb else "bend"

print(f"capacity(2,16) = {capacity(2, 16, SY, K_LTB, CS):.0f} N, "
      f"dominant-mode proxy = {gov_mode(2, 16, SY, K_LTB, CS)}")

## 7. Diagnose the model before fitting it

The plots below use one common scale. Point color is the predicted dominant
mode; marker shape is the observed note category. Beam IDs let you connect
every miss to the full note table.

In [ ]:
def observed_note_class(note):
    s = str(note).lower()
    if "separ" in s or "peel" in s:
        return "flange-web separation"
    if any(word in s for word in ("flip", "tip", "twist", "buckl")):
        return "tip/twist"
    if "fract" in s or "vertical" in s:
        return "fracture"
    return "other"

MODE_ORDER = ["bend", "interaction/shear proxy", "LTB"]
MODE_COLOR = {"bend": "tab:blue", "interaction/shear proxy": "tab:orange", "LTB": "tab:red"}
OBS_MARKER = {"fracture": "o", "flange-web separation": "s", "tip/twist": "^", "other": "D"}
from matplotlib.lines import Line2D

def diagnostic_plots(sy, k, cs, label):
    d = df.copy()
    d["predicted_N"] = [capacity(b, H, sy, k, cs) for b, H in zip(d.b, d.H)]
    d["mode_proxy"] = [gov_mode(b, H, sy, k, cs) for b, H in zip(d.b, d.H)]
    d["observed_class"] = d.failure_note.map(observed_note_class)
    d["residual_pct"] = 100*(d.predicted_N/d.strength_N - 1)
    hi = 1.08*max(d.predicted_N.max(), d.strength_N.max())
    fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharex=True, sharey=True)
    panels = [("all beams", d)] + [(mode, d[d.mode_proxy == mode]) for mode in MODE_ORDER]
    for ax, (title, sub) in zip(axes, panels):
        for _, row in sub.iterrows():
            ax.scatter(row.predicted_N, row.strength_N, s=70,
                       c=MODE_COLOR[row.mode_proxy],
                       marker=OBS_MARKER[row.observed_class], edgecolor="k")
            ax.annotate(str(int(row.beam_id)), (row.predicted_N, row.strength_N),
                        xytext=(4, 3), textcoords="offset points", fontsize=8)
        ax.plot([0, hi], [0, hi], "k--", alpha=0.5)
        ax.set_title(title); ax.set_xlabel("predicted [N]")
        ax.grid(alpha=0.25); ax.set_xlim(0, hi); ax.set_ylim(0, hi)
        if sub.empty:
            ax.text(0.5, 0.5, "no beams", transform=ax.transAxes, ha="center")
    axes[0].set_ylabel("measured [N]")
    legend_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=color,
               markeredgecolor="k", label=f"predicted: {mode}")
        for mode, color in MODE_COLOR.items()
    ] + [
        Line2D([0], [0], marker=marker, color="k", linestyle="none",
               markerfacecolor="white", label=f"observed: {obs}")
        for obs, marker in OBS_MARKER.items()
    ]
    axes[0].legend(handles=legend_handles, fontsize=6, loc="lower right")
    fig.suptitle(f"{label}: sigma_y={sy/1e6:.1f} MPa, k={k:.3f}, c_s={cs:.2f}")
    plt.tight_layout(); plt.show()
    cols = ["beam_id", "b", "H", "strength_N", "predicted_N", "residual_pct",
            "mode_proxy", "observed_class", "failure_note"]
    print(d[cols].round({"b": 2, "H": 2, "strength_N": 1,
                         "predicted_N": 1, "residual_pct": 1}).to_string(index=False))
    mape = d.residual_pct.abs().mean()
    print(f"\n{label} MAPE = {mape:.1f}%")
    return d, float(mape)

diag_nominal, mape_nom = diagnostic_plots(SY, K_LTB, CS, "Nominal handbook parameters")
df["cap_nominal"] = diag_nominal.predicted_N
print(f"CHECKPOINT: nominal error should be about 14% MAPE.")

## 8. Tune by hand before optimizing

Change the three values and rerun this cell. Try to improve the parity plot,
but watch which failure notes and predicted modes each change helps or hurts.

- `TRY_SY_MPA` moves flexural and average-shear strength together.
- `TRY_K` mostly moves the LTB-limited beams.
- `TRY_CS` moves only the average-shear side of the empirical interaction.

In [ ]:
TRY_SY_MPA = 76.0    # edit
TRY_K = 0.33         # edit
TRY_CS = 1.00        # edit

diag_try, mape_try = diagnostic_plots(
    TRY_SY_MPA*1e6, TRY_K, TRY_CS, "Your trial parameters")

## 9. Calibrate the three parameters

Now automate the search. The loss is mean squared log-error, so comparable
percentage misses receive comparable weight:

$$L(\sigma_y,k,c_s)=\frac{1}{14}\sum_{i=1}^{14}
\left(\ln P_{pred,i}-\ln P_{meas,i}\right)^2.$$

Fill in the loss. A coarse search and Nelder-Mead polish are provided.

In [ ]:
from scipy.optimize import minimize

def loss(theta):
    sy, k, cs = theta
    if not (30e6 < sy < 150e6 and 0.05 < k < 1.5 and 0.2 < cs < 4):
        return 1e9
    pred = np.array([capacity(b, H, sy, k, cs) for b, H in zip(df.b, df.H)])
    return ____    # >>> FILL IN: mean squared difference of log predictions and measurements

grid = [(sy, k, cs) for sy in np.linspace(55e6, 100e6, 19)
        for k in np.linspace(0.10, 0.90, 17) for cs in (0.6, 0.8, 1.0, 1.3, 1.7)]
theta0 = min(grid, key=loss)
res = minimize(loss, theta0, method="Nelder-Mead",
               options=dict(xatol=1e-4, fatol=1e-10, maxiter=4000))
SY_CAL, K_CAL, CS_CAL = res.x
diag_cal, mape_cal = diagnostic_plots(SY_CAL, K_CAL, CS_CAL, "Calibrated parameters")
df["cap_cal"] = diag_cal.predicted_N
print(f"calibrated: sigma_y = {SY_CAL/1e6:.1f} MPa, k = {K_CAL:.3f}, cs = {CS_CAL:.2f}")
print(f"CHECKPOINT: sigma_y = 66.5 MPa, k = 0.377, cs = 2.25, "
      f"error = 4.2% MAPE. Your error: {mape_cal:.1f}%.")

Do not read a fitted parameter as a direct material measurement. Decide
whether each change corrects a plausible numerical assumption or compensates
for missing physics. The failure-note table is evidence for that distinction.

## 10. Optimize the equation design

Search the design box for the best predicted strength-to-weight under the
calibrated empirical model. Fill in the objective.

In [ ]:
best = (None, None, -1)
for b in np.arange(1.25, 7.01, 0.05):
    for H in np.arange(5.0, 16.01, 0.05):
        cap = capacity(b, H, SY_CAL, K_CAL, CS_CAL)
        sw = ____    # >>> FILL IN: predicted capacity per estimated gram
        if sw > best[2]:
            best = (round(b, 2), round(H, 2), sw)
b_eq, H_eq, sw_eq = best
mode_cal = gov_mode(b_eq, H_eq, SY_CAL, K_CAL, CS_CAL)
mode_nom = gov_mode(b_eq, H_eq, SY, K_LTB, CS)
print(f"EQUATION DESIGN: b={b_eq} mm, H_web={H_eq} mm, predicted {sw_eq:.1f} N/g")
print(f"dominant-mode proxy at this geometry: calibrated={mode_cal}, nominal={mode_nom}")
print("CHECKPOINT: b = 1.25, H_web = 13.40, predicted 46.6 N/g.")

Staff query one common equation design through the frozen oracle. The
returned strength appears at the start of Pre-lab 2.

## Memo questions

1. Compare measured and estimated masses. Name one physical reason they differ,
   and explain why the pre-print optimizer still uses estimated mass.
2. Which dominant-mode proxy applies to the equation design under calibrated
   and nominal parameters? Use the final cell and explain what moved.
3. For $\sigma_y$, `k`, and `c_s`, decide whether calibration is a correction
   to a number or a confession of missing physics. Cite specific residuals and
   failure notes from the diagnostic table.
4. The optimum sits at minimum `b`, where predicted str/w still increases as
   the web thins. Which observed failures should reduce your trust there?

Use the final design output, the nominal/calibrated plots, and beam IDs with
their full failure notes. No additional optimization code is required.